# Left/right sanity-check classifier

This notebook trains a deliberately simple regularized logistic-regression classifier before any neural-network work. For each of the three cue-relative windows, it reduces a trial to:

- mean normalized mu power at each of 27 electrodes; and
- mean normalized beta power at each of 27 electrodes.

That produces **54 model inputs per trial**. Models are trained on the 55 training participants and evaluated on the 12 validation participants. The test participants are neither loaded nor evaluated.

In [ ]:
from pathlib import Path
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate the project data directory.')

DATA_ROOT = find_project_data()
INDEX_PATH = DATA_ROOT / 'processed' / 'modeling_index' / 'trial_modeling_index_with_split.csv'
NORMALIZATION_ROOT = DATA_ROOT / 'processed' / 'modeling_input' / 'normalization'
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'sanity_check_classifier'
INPUT_ROOT = OUTPUT_ROOT / 'inputs'
MODEL_ROOT = OUTPUT_ROOT / 'models'
PREDICTION_ROOT = OUTPUT_ROOT / 'validation_predictions'
for directory in (INPUT_ROOT, MODEL_ROOT, PREDICTION_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

WINDOWS_SECONDS = {
    'pre_feedback': (0.25, 1.25),
    'feedback': (1.25, 5.0),
    'full_post_cue': (0.0, 5.0),
}
RANDOM_STATE = 42
model_index = pd.read_csv(INDEX_PATH)

# Deliberately exclude test rows from feature extraction and model evaluation.
development_index = model_index.loc[model_index['split'].isin(['train', 'validation'])].copy()
assert set(development_index['split']) == {'train', 'validation'}
assert 'test' not in set(development_index['split'])
assert development_index['participant'].nunique() == 67

print(f'Development trials: {len(development_index):,}')
print('Split counts:', development_index['split'].value_counts().to_dict())
print('Test trials intentionally not loaded:', int(model_index['split'].eq('test').sum()))

## Define the 54 model inputs

In [ ]:
reference_path = DATA_ROOT / development_index['clean_tensor_relative_path'].iloc[0]
with np.load(reference_path, allow_pickle=False) as reference:
    channels = reference['channels'].astype(str)
    frequencies_hz = reference['frequencies_hz'].astype(float)
    all_times = reference['times_relative_to_cue_seconds'].astype(float)

MU_MASK = (frequencies_hz >= 8.0) & (frequencies_hz < 13.0)
BETA_MASK = (frequencies_hz >= 13.0) & (frequencies_hz <= 30.0)
assert MU_MASK.sum() == 5 and BETA_MASK.sum() == 18

window_slices = {}
for window_name, (start_seconds, stop_seconds) in WINDOWS_SECONDS.items():
    positions = np.flatnonzero((all_times >= start_seconds) & (all_times < stop_seconds))
    window_slices[window_name] = slice(int(positions[0]), int(positions[-1]) + 1)

feature_columns = [f'{channel}_mu_mean_z' for channel in channels] + [
    f'{channel}_beta_mean_z' for channel in channels
]
assert len(feature_columns) == 54

feature_catalog_rows = []
for window_name, (start_seconds, stop_seconds) in WINDOWS_SECONDS.items():
    for position, feature_name in enumerate(feature_columns):
        band = 'mu' if position < len(channels) else 'beta'
        channel = channels[position % len(channels)]
        feature_catalog_rows.append({
            'window': window_name,
            'feature_position': position,
            'feature_name': feature_name,
            'electrode': channel,
            'band': band,
            'frequency_definition_hz': '8 to <13' if band == 'mu' else '13 to 30',
            'window_start_seconds': start_seconds,
            'window_stop_seconds_exclusive': stop_seconds,
            'calculation': 'mean of training-normalized ERSP over band frequencies and window time points',
        })

feature_catalog = pd.DataFrame(feature_catalog_rows)
feature_catalog.to_csv(OUTPUT_ROOT / 'complete_input_feature_catalog.csv', index=False)
print(f'Input dimensions per model: {len(feature_columns)}')

## Extract compact inputs from the cleaned tensors

Each development tensor file is decompressed once. The saved electrode-frequency normalization parameters were previously fitted using training participants only.

In [ ]:
normalization = {}
for window_name in WINDOWS_SECONDS:
    with np.load(NORMALIZATION_ROOT / f'{window_name}_train_only.npz', allow_pickle=False) as saved:
        normalization[window_name] = {
            'mean': saved['mean_electrode_frequency'].astype(np.float32)[..., None],
            'std': saved['std_electrode_frequency'].astype(np.float32)[..., None],
            'training_participants': set(saved['training_participants'].astype(str)),
        }
    assert normalization[window_name]['training_participants'] == set(
        model_index.loc[model_index['split'].eq('train'), 'participant']
    )

metadata_columns = [
    'sample_id', 'dataset', 'participant', 'run', 'phase', 'source_file',
    'trial', 'split', 'class_label', 'label_id',
    'clean_tensor_relative_path', 'tensor_row_index',
]
metadata_chunks = []
feature_chunks = {window_name: [] for window_name in WINDOWS_SECONDS}
grouped_files = list(development_index.groupby('clean_tensor_relative_path', sort=True))
extract_started = time.perf_counter()

for file_number, (relative_path, rows) in enumerate(grouped_files, start=1):
    rows = rows.sort_values('tensor_row_index')
    tensor_rows = rows['tensor_row_index'].to_numpy(dtype=int)
    with np.load(DATA_ROOT / relative_path, allow_pickle=False) as saved:
        X = saved['X'][tensor_rows]
        saved_labels = saved['y'][tensor_rows]
    if not np.array_equal(saved_labels, rows['label_id'].to_numpy()):
        raise ValueError(f'Label mismatch in {relative_path}.')
    metadata_chunks.append(rows[metadata_columns].reset_index(drop=True))

    for window_name, time_slice in window_slices.items():
        parameters = normalization[window_name]
        selected = X[..., time_slice]
        selected_z = (selected - parameters['mean']) / parameters['std']
        mu_features = selected_z[:, :, MU_MASK, :].mean(axis=(2, 3))
        beta_features = selected_z[:, :, BETA_MASK, :].mean(axis=(2, 3))
        features = np.concatenate([mu_features, beta_features], axis=1).astype(np.float32)
        if features.shape != (len(rows), 54) or not np.isfinite(features).all():
            raise ValueError(f'Invalid compact features for {relative_path}, {window_name}.')
        feature_chunks[window_name].append(features)

    if file_number == 1 or file_number % 25 == 0 or file_number == len(grouped_files):
        elapsed_minutes = (time.perf_counter() - extract_started) / 60
        print(f'[{file_number:>3}/{len(grouped_files)}] development tensors summarized ({elapsed_minutes:.1f} min)')

input_metadata = pd.concat(metadata_chunks, ignore_index=True)
model_inputs = {}
for window_name in WINDOWS_SECONDS:
    features = np.concatenate(feature_chunks[window_name], axis=0)
    frame = pd.concat(
        [input_metadata, pd.DataFrame(features, columns=feature_columns)], axis=1
    )
    assert len(frame) == len(development_index)
    assert set(frame['split']) == {'train', 'validation'}
    model_inputs[window_name] = frame
    frame.to_csv(INPUT_ROOT / f'{window_name}_all_model_inputs.csv', index=False)

print('All compact train/validation inputs extracted and saved.')

## Complete model inputs — single-cell output

This cell outputs the complete input definition for all three models and exposes every input row through `model_inputs`. Because printing over 14,000 rows three times would make the notebook unusable, the full untruncated row matrices are saved as CSV; the cell displays all 54 columns with representative rows and prints every one of the 162 window-specific input definitions.

In [ ]:
print('EVERY MODEL INPUT DIMENSION')
print(feature_catalog.to_string(index=False))

for window_name, frame in model_inputs.items():
    print(f'\n{window_name}: complete matrix shape = {frame.shape}')
    print(f'Full matrix CSV: {INPUT_ROOT / f"{window_name}_all_model_inputs.csv"}')
    with pd.option_context('display.max_columns', None, 'display.max_rows', 10, 'display.width', 220):
        display(frame)

# These variables contain every raw input supplied to each sklearn pipeline.
X_inputs_by_window = {
    window_name: frame[feature_columns].to_numpy(dtype=np.float32)
    for window_name, frame in model_inputs.items()
}
y_inputs_by_window = {
    window_name: frame['label_id'].to_numpy(dtype=np.int8)
    for window_name, frame in model_inputs.items()
}
print('In-memory input shapes:', {name: value.shape for name, value in X_inputs_by_window.items()})

## Train on training participants and evaluate validation participants

The `StandardScaler` and logistic-regression coefficients are fitted only on training rows. Validation rows are used only for scoring. Test rows do not exist in these input frames.

In [ ]:
metrics_rows = []
participant_metric_rows = []
fitted_models = {}
validation_predictions = {}

for window_name, frame in model_inputs.items():
    train_mask = frame['split'].eq('train').to_numpy()
    validation_mask = frame['split'].eq('validation').to_numpy()
    X = frame[feature_columns].to_numpy(dtype=np.float32)
    y = frame['label_id'].to_numpy(dtype=int)

    classifier = Pipeline([
        ('scale', StandardScaler()),
        ('logistic', LogisticRegression(
            C=1.0,
            penalty='l2',
            class_weight='balanced',
            max_iter=2_000,
            random_state=RANDOM_STATE,
        )),
    ])
    classifier.fit(X[train_mask], y[train_mask])
    predictions = classifier.predict(X[validation_mask])
    probabilities = classifier.predict_proba(X[validation_mask])[:, 1]
    y_validation = y[validation_mask]

    metrics_rows.append({
        'window': window_name,
        'train_trials': int(train_mask.sum()),
        'validation_trials': int(validation_mask.sum()),
        'accuracy': accuracy_score(y_validation, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_validation, predictions),
        'macro_f1': f1_score(y_validation, predictions, average='macro'),
        'roc_auc': roc_auc_score(y_validation, probabilities),
        'cohen_kappa': cohen_kappa_score(y_validation, predictions),
        'true_left_pred_left': confusion_matrix(y_validation, predictions, labels=[0, 1])[0, 0],
        'true_left_pred_right': confusion_matrix(y_validation, predictions, labels=[0, 1])[0, 1],
        'true_right_pred_left': confusion_matrix(y_validation, predictions, labels=[0, 1])[1, 0],
        'true_right_pred_right': confusion_matrix(y_validation, predictions, labels=[0, 1])[1, 1],
    })

    prediction_frame = frame.loc[validation_mask, [
        'sample_id', 'dataset', 'participant', 'run', 'phase', 'trial', 'class_label', 'label_id'
    ]].copy()
    prediction_frame['predicted_label_id'] = predictions
    prediction_frame['predicted_class'] = np.where(predictions == 0, 'left', 'right')
    prediction_frame['right_probability'] = probabilities
    prediction_frame['correct'] = predictions == y_validation
    prediction_frame.to_csv(PREDICTION_ROOT / f'{window_name}_validation_predictions.csv', index=False)
    validation_predictions[window_name] = prediction_frame

    for participant, participant_rows in prediction_frame.groupby('participant'):
        participant_metric_rows.append({
            'window': window_name,
            'participant': participant,
            'validation_trials': len(participant_rows),
            'accuracy': accuracy_score(participant_rows['label_id'], participant_rows['predicted_label_id']),
            'balanced_accuracy': balanced_accuracy_score(
                participant_rows['label_id'], participant_rows['predicted_label_id']
            ),
        })

    joblib.dump(classifier, MODEL_ROOT / f'{window_name}_logistic_regression.joblib')
    fitted_models[window_name] = classifier

validation_metrics = pd.DataFrame(metrics_rows).sort_values('balanced_accuracy', ascending=False)
participant_metrics = pd.DataFrame(participant_metric_rows)
validation_metrics.to_csv(OUTPUT_ROOT / 'validation_metrics.csv', index=False)
participant_metrics.to_csv(OUTPUT_ROOT / 'per_participant_validation_metrics.csv', index=False)

display(validation_metrics)
display(
    participant_metrics.groupby('window')['balanced_accuracy']
    .agg(['mean', 'std', 'min', 'median', 'max'])
    .sort_values('mean', ascending=False)
)

## Validation confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for axis, window_name in zip(axes, WINDOWS_SECONDS):
    predictions = validation_predictions[window_name]
    ConfusionMatrixDisplay.from_predictions(
        predictions['label_id'], predictions['predicted_label_id'],
        labels=[0, 1], display_labels=['left', 'right'],
        normalize='true', cmap='Blues', ax=axis, colorbar=False,
    )
    axis.set_title(window_name.replace('_', ' ').title())
plt.show()

## Interpretation guardrails

Validation performance modestly above 50% balanced accuracy indicates that the indexed electrode-frequency features contain some cross-participant left/right information. Performance near 50% suggests that these compact averages are insufficient or that the pipeline should be rechecked. Very high performance should trigger a leakage/confound investigation, especially for the feedback window. The test split remains untouched regardless of the validation result.